# CS-MAP FINAL EXPERIMENTS

## 0. Main setup

### 0.1. Import libraries and set RNG seeds

In [ ]:
import os
from itertools import combinations
from random import sample, seed
import warnings
import matplotlib.pyplot as plt

from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

import numpy as np
from numpy.random import default_rng
import matplotlib.pyplot as plt
from scipy.optimize import minimize, root, least_squares, Bounds, LinearConstraint
from scipy.stats import pearsonr
from scipy.special import factorial, stirling2
from scipy.linalg import eig, inv, expm, LinAlgError

import pandas as pd

from lifelines import KaplanMeierFitter
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

warnings.filterwarnings('ignore', category=RuntimeWarning)
np.seterr(all='ignore')

RANDOM_SEED = 123456

### 0.2. Define functions

#### 0.2.1 Utility functions

In [ ]:
def parameter_swap(parameters):
    """
    Converts between {alpha0, D0, D1} and {alpha0, rates, P0, P1}. The input dict must contain either the pair (D0,D1) or (rates,P0,P1).
    """
    if "D0" in parameters and "D1" in parameters:
        # Change (alpha0, D0, D1) into (alpha0, lambda, P0, P1)
        D0, D1 = parameters["D0"], parameters["D1"]
        rates = -np.diag(D0)
        inverse_rates = np.diag(1.0/rates)
        P0 = inverse_rates @ D0 + np.eye(D0.shape[0])
        P1 = inverse_rates @ D1
        return {"alpha0": parameters.get("alpha0"), "rates": rates, "P0": P0, "P1": P1}
    
    elif "rates" in parameters and "P0" in parameters and "P1" in parameters:
        # Change (alpha0, lambda, P0, P1) into (alpha0, D0, D1)
        rates, P0, P1 = parameters["rates"], parameters["P0"], parameters["P1"]
        D0 = np.diag(rates) @ (P0 - np.eye(P0.shape[0]))
        D1 = np.diag(rates) @ P1
        return {"alpha0": parameters.get("alpha0"), "D0": D0, "D1": D1}
    
    else:
        # Errors exit
        raise ValueError("parameter dict must contain either D0 & D1 "
                         "or rates, P0 & P1")

def draw_CSMAP_parameters(order, distribution, parameter_A=2, parameter_B=0.5, random_seed=None):
    """
    Returns a dictionary {alpha0, D0, D1}  for an order x order CS-MAP sampled as follows:

      alpha0 ~ Dirichlet(1, ..., 1) of length order
      lambda_i ~ Gamma(gamma_shape, gamma_scale), for each i, independently
      for each row i:
         y ~ Dirichlet(1, ... ,1)   of length 2m-1
         split  y ->  y0_off (length order-1)  |  y1 (length order)
         place y0_off into the off-diagonal element of P0[i], set P0[i,i]=0
         P1[i,:] = y1
         renormalise so that P0[i, :]+ P1[i, :] == 1
    """
    # Set the random number generator, for reproducibility
    if random_seed is not None:
        rng = default_rng(random_seed)
    else:
        rng = default_rng() # fallback to non-deterministic

    # Sample the parameters
    alpha0 = rng.dirichlet(np.ones(order))
    if distribution=="gamma":
        rates = rng.gamma(parameter_A, parameter_B, size=order)
    elif distribution=="uniform":
        rates = rng.uniform(parameter_A, parameter_B, order)
    else:
        print("Enter a valid distribution for the rates vector: \"gamma\" or \"uniform\".")
    P0 = np.zeros((order, order))
    P1 = np.zeros((order, order))
    for i in range(order):
        y  = rng.dirichlet(np.ones(2 * order - 1))
        y0 = y[:order-1]           # off-diagonal probabilities for P0
        y1 = y[order-1:]           # full row for P1

        P0[i, :i] = y0[:i]
        P0[i, i+1:] = y0[i:]
        P1[i, :] = y1

        row_sum = P0[i].sum() + P1[i].sum()
        P0[i] /= row_sum
        P1[i] /= row_sum
        # P0[i,i] stays 0 by construction

    parameters = parameter_swap({"alpha0": alpha0, "rates": rates, "P0": P0, "P1": P1})
    return parameters

#### 0.2.2. Simulation and censorship functions

In [ ]:
def simulate_csmap(params, kmax, N_sim=1000, seed=RANDOM_SEED, progress=False):
    """
    Simulates inter-event times from a CS-MAP up to a maximum number of events (kmax), and returns results in wide-form pandas DataFrame.

    Parameters:
    - params: dictionary with CS-MAP parameters (alpha0, D0, D1)
    - kmax: maximum number of events (inter-event times)
    - N_sim: number of CS-MAP runs
    - seed: random seed for reproducibility
    - progress: if True, shows the current iteration on execution

    Returns:
    - A dictionary with:
        - 'id': array of simulation indices
        - 'Tn': (N_sim, kmax) array of inter-event times
        - 'N': array of number of observed events before termination
        - 'last_state': array indicating terminal state (0 = censoring, 1 = death)
    """
    # Fix the random seed
    if seed is not None:
        np.random.seed(seed)

    # Extract the CS-MAP parameters
    alpha0 = params["alpha0"]
    D0 = params["D0"]
    D1 = params["D1"]
    m = len(alpha0)

    # Normalize transition matrices
    lambda_rates = -np.diag(D0)
    P0 = D0 / lambda_rates[:, None]
    np.fill_diagonal(P0, 0)
    P1 = D1 / lambda_rates[:, None]

    # Initialize storage
    Tn = -np.ones((N_sim, kmax))
    N = -np.ones(N_sim, dtype=int)
    last_state = np.zeros(N_sim, dtype=int)

    # Run all of the simulations
    for i in range(N_sim):
        state = np.random.choice(m, p=alpha0)
        for j in range(kmax):
            t = 0.0
            while state < m:
                t += np.random.exponential(1 / lambda_rates[state]) # NumPy.random.exponential USES SCALE == 1/RATE AS ARGUMENT
                probs = np.concatenate([P0[state], P1[state]])
                probs = probs / probs.sum() # TO ACCOUNT FOR NUMERICAL ERRORS!!!!
                state = np.random.choice(2 * m, p=probs)
            Tn[i, j] = t
            # Check if the death state is visited
            if state == 2 * m - 1:
                N[i] = j + 1
                last_state[i] = 1
                break
            # If not death, a recurrence state has been visited
            else:
                state -= m
            # Assigns a censorship if kmax events have occurred (note that last_state[i]==0 already)
            if j == kmax:
                N[i] = kmax
            
        # Display progress
        if progress == True and (i+1)%(N_sim//10) == 0:
            print(f"{100*(i+1)//N_sim}% of the simulations finished!")

    # Construct the DataFrame
    dataframe = pd.DataFrame(Tn[:, :max(N)], columns=[f"T{i+1}" for i in range(max(N))])
    dataframe["N"] = N
    dataframe["last_state"] = last_state

    return dataframe

def apply_random_censoring(
    df: pd.DataFrame,
    how_many,
    *,
    censor_rate: float = 0.1,
    epsilon: float = 1e-6,
    integer_mode: str | bool = "auto",
    seed: int | None = None,
) -> pd.DataFrame:
    """
    Randomly right-censor *some* individuals in an RD CS-MAP dataset (wide format).

    Input dataset columns:
      - T1, T2, ..., Tk   (inter-event gaps; positive if recorded, -1 if unrealized)
      - N                  (number of observed gaps up to and including the last gap)
      - last_state         (1 = death observed; 0 = censored)
    Output dataset has the same schema, with some formerly-uncensored individuals now censored.

    Parameters
    ----------
    df : DataFrame
        RD CS-MAP dataset in wide form.
    how_many : int or float
        If int: the exact number of *currently uncensored* individuals to censor (uniform at random).
        If float in (0,1]: the fraction of the *currently uncensored* to censor.
    censor_rate : float
        Rate parameter λ of the exponential distribution.
        Mean censoring time is 1/λ.
    epsilon : float
        Lower bound for continuous censoring times at the selected “death gap”.
        (Used only when integer_mode=False; ensures strictly positive censoring times.)
    integer_mode : {"auto", True, False}
        - "auto": enable integer mode iff all positive times across T-columns are integers.
        - True:  force integer censoring (use integer times).
        - False: use continuous times.
    seed : int or None
        Random seed (all randomness controlled by this).

    Censoring mechanism for each selected row i (with original N_i, last_state_i == 1):
      1) Choose a new N_i^cens uniformly from {1, 2, ..., N_i}. (Earlier gaps = heavier censoring.)
      2) Set all times beyond N_i^cens to -1.
      3) Replace the gap T_{N_i^cens} by a *censoring time*:
           - continuous mode: Uniform(epsilon, old_T_{N_i^cens}); if old_T <= epsilon, use epsilon
           - integer mode   : random integer in [1, max(old_T - 1, 1)] to avoid tying a true event
      4) Set last_state_i := 0.

    Returns
    -------
    DataFrame
        A *new* DataFrame with synthetic censoring applied to the selected individuals.
    """
    rng = np.random.default_rng(seed)
    out = df.copy(deep=True)

    # -------- find T-columns and basic checks --------
    tcols = sorted([c for c in out.columns if c.startswith("T") and c[1:].isdigit()],
                   key=lambda c: int(c[1:]))
    if not tcols:
        raise ValueError("No T-columns found (expected T1, T2, ...).")

    # enforce numeric in T-columns
    for c in tcols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(-1.0)

    # integer-mode auto detection (dataset-wide)
    if integer_mode == "auto":
        pos_vals = np.concatenate([out[c].to_numpy() for c in tcols])
        pos_vals = pos_vals[pos_vals > 0]
        integer_mode = bool(pos_vals.size and np.allclose(pos_vals, np.round(pos_vals), rtol=0, atol=1e-12))

    # -------- choose who to censor (uniform among currently uncensored) --------
    uncensored_mask = (out["last_state"] == 1)
    uncensored_idx = np.flatnonzero(uncensored_mask.to_numpy())
    m_unc = len(uncensored_idx)

    if isinstance(how_many, float):
        if not (0 < how_many <= 1):
            raise ValueError("If 'how_many' is a float, it must be in (0,1].")
        n_to_censor = int(np.floor(how_many * m_unc))
    else:
        n_to_censor = int(how_many)

    if n_to_censor < 0:
        raise ValueError("'how_many' must be nonnegative.")
    if n_to_censor > m_unc:
        raise ValueError(f"Requested to censor {n_to_censor} but only {m_unc} uncensored individuals are available.")
    if n_to_censor == 0:
        return out

    chosen = rng.choice(uncensored_idx, size=n_to_censor, replace=False)

    # -------- perform censoring gap-by-gap on chosen rows --------
    for i in chosen:
        N_old = int(out.at[i, "N"])
        if N_old < 1:
            # nothing to do: degenerate row
            continue

        # Choose new N uniformly from {1,...,N_old}
        N_new = int(rng.integers(1, N_old + 1))  # upper is exclusive
        # Get the gap column name where we will place the censoring time
        gap_col = f"T{N_new}"
        if gap_col not in out.columns:
            # if for some reason the column is missing, create it
            out[gap_col] = -1.0

        # Old gap time at the censoring gap (may be a true event gap in the original)
        old_t = float(out.at[i, gap_col])
        if not np.isfinite(old_t) or old_t <= 0:
            # Fallback: if an unexpected nonpositive time is found, set a small positive default
            old_t = 1.0 if integer_mode else max(epsilon, 1.0)

        # Choose a censoring time for that gap
        if integer_mode:
            # choose from 1..max(old_t-1,1) to avoid tying a true event when old_t>1
            hi = int(np.floor(old_t)) - 1
            cens_t = 1 if hi < 1 else int(rng.integers(1, hi + 1))
        else:
            # Exponential censoring truncated at old_t
            cens_t = rng.exponential(scale=1.0/censor_rate)
            if cens_t >= old_t:
                cens_t = old_t - epsilon
            if cens_t <= 0:
                cens_t = epsilon

        # Write the censoring time
        out.at[i, gap_col] = cens_t

        # Set all *later* gaps to unrealized (-1)
        for j in range(N_new + 1, len(tcols) + 1):
            colj = f"T{j}"
            if colj in out.columns:
                out.at[i, colj] = -1.0

        # Update N and last_state
        out.at[i, "N"] = N_new
        out.at[i, "last_state"] = 0

    # If integer mode, coerce T-columns back to int where positive
    if integer_mode:
        for c in tcols:
            arr = out[c].to_numpy()
            pos = arr > 0
            arr[pos] = np.rint(arr[pos]).astype(int)
            out[c] = arr.astype(int)

    return out

def km_impute_times(
    df: pd.DataFrame,
    time_col: str = "time",
    status_col: str = "last_state",
    integer_mode: bool = False,
    show_log: bool = False,
) -> pd.Series:
    """
    KM-based single-value imputation of survival times.

    For censored rows:
      - if censor time c < last observed event time t_max:
            impute with KM conditional mean E[T | T > c]
      - else (c >= t_max):
            HEURISTIC: impute time = 1.33 * c

    For event rows: return the observed time.

    Parameters
    ----------
    df           : DataFrame with columns `time_col` and `status_col`
    time_col     : name of the time-to-event/censoring column
    status_col   : name of the event indicator (nonzero = event, 0 = right-censored)
    integer_mode : if True, round *imputed* values to nearest integer (events untouched)
    show_log     : if True, print a per-row method summary

    Returns
    -------
    pd.Series of imputed times (same index as `df`)
    """
    # ---- 0) pull vectors
    times  = df[time_col].to_numpy(dtype=float)
    events = (df[status_col].to_numpy() != 0).astype(int)

    # ---- 1) fit KM
    kmf = KaplanMeierFitter().fit(times, event_observed=events)

    # KM step locations (event times) and survival after each step
    t_km = kmf.survival_function_.index.values.astype(float)        # sorted unique *event* times
    S_km = kmf.survival_function_["KM_estimate"].to_numpy()

    # survival just *before* each event time
    S_prev = np.concatenate(([1.0], S_km[:-1]))                     # Ŝ(t_j-)

    # Ŝ(t_j-) - Ŝ(t_j)
    dF = S_prev - S_km  # same length as t_km

    # last observed event time
    t_max = float(np.max(times[events == 1])) if events.sum() > 0 else 0.0

    # ---- 2) per-row imputation
    out = times.copy()
    methods = np.array(["observed"] * len(df), dtype=object)

    if events.sum() == 0:
        # no events → everything is “beyond t_max”; apply 1.33×c
        out[:] = 1.33 * times
        methods[:] = "heuristic_133x"
    else:
        cens_idx = np.where(events == 0)[0]
        for i in cens_idx:
            c = times[i]
            if c >= t_max:
                # no events after c → 1.33xc
                out[i] = 1.33 * c
                methods[i] = "heuristic_133x"
            else:
                # conditional mean beyond c under KM
                # position of first event STRICTLY after c
                pos = np.searchsorted(t_km, c, side="right")
                # survival just BEFORE c
                left = np.searchsorted(t_km, c, side="left")
                Sc = S_prev[left] if left < len(S_prev) else S_km[-1]
                
                # numerical guard (shouldn't be 0 when c < t_max)
                if Sc <= 0 or pos >= len(t_km):
                    out[i] = 1.33 * c
                    methods[i] = "heuristic_133x"
                # E[T | T>c] = sum_{j>=pos} t_j * ΔF_j / S(c-)
                else:
                    num = np.sum(t_km[pos:] * dF[pos:])
                    out[i] = num / Sc if num > 0 else 1.33 * c
                    methods[i] = "km_cond_mean" if num > 0 else "heuristic_133x"

    # ---- 3) integer rounding (only for imputed values)
    if integer_mode:
        imputed_mask = (methods != "observed")
        out[imputed_mask] = np.rint(out[imputed_mask])

    # ---- 4) optional log
    if show_log:
        log_df = pd.DataFrame({
            "time": times,
            "event": events,
            "method": methods,
            "imputed_value": out
        }, index=df.index)
        print(log_df)

    return pd.Series(out, index=df.index, name=f"{time_col}_imputed")

def complete_rd_csmap_dataset(
    df: pd.DataFrame,
    show_log: bool = False,
) -> pd.DataFrame:
    """
    Complete an RD CS-MAP wide dataset (T1..Tk, N, last_state) and RETURN a
    *standard* RD CS-MAP dataset with the same labels:
        T1..Tmax , N , last_state (=1 for all rows)

    Steps
    -----
    1) Impute r = N-1 via KM conditional mean on the discrete count using
       km_impute_times with integer_mode=True; then set N := r_hat + 1
       (and guard r_hat >= r_obs).
    2) For each gap Tn:
       - Fit KM at gap n using rows that reached n and have positive Tn.
       - Impute censored-at-gap rows directly with km_impute_times on the
         reached-n sample (observed rows are left unchanged).
       - If the gap did not exist but N_hat >= n, create a new Tn:
           • Prefer unconditional KM mean for Tn (fit on reached-n).
           • Else fall back to the individual's mean of existing positive gaps;
             else global median of all positive times.
    3) Set last_state = 1 for all rows; set Tn = 0 for columns beyond N.

    Returns
    -------
    DataFrame with columns T1..Tmax, N, last_state (all ones).
    """
    res = df.copy()

    # --- Ensure numeric & standard form ---
    res["N"] = pd.to_numeric(res["N"], errors="coerce").fillna(0).astype(int)
    res["last_state"] = pd.to_numeric(res["last_state"], errors="coerce").fillna(0).astype(int)

    tcols = sorted([c for c in res.columns if c.startswith("T") and c[1:].isdigit()],
                   key=lambda x: int(x[1:]))
    for c in tcols:
        res[c] = pd.to_numeric(res[c], errors="coerce").fillna(0.0)

    N_obs  = res["N"].to_numpy().astype(int)
    ls_obs = res["last_state"].to_numpy().astype(int)   # 0=cens, 1=death

    # ---------- A) Impute r = N-1 and set N_hat ----------
    res["r_obs"] = np.maximum(N_obs - 1, 0)
    r_df = res.rename(columns={"r_obs": "time", "last_state": "last_state"})
    r_hat = km_impute_times(
        r_df[["time", "last_state"]],
        time_col="time",
        status_col="last_state",
        integer_mode=True,
        show_log=show_log
    ).to_numpy().astype(int)

    # Guard monotonicity (don’t reduce r below observed r)
    r_hat = np.maximum(r_hat, res["r_obs"].to_numpy().astype(int))
    N_hat = r_hat + 1

    max_n = int(N_hat.max())

    # Precompute a global median of all positive gap times (last resort)
    all_pos = []
    for c in tcols:
        v = res[c].to_numpy()
        all_pos.append(v[v > 0])
    all_pos = np.concatenate(all_pos) if len(all_pos) else np.array([])
    global_median = float(np.median(all_pos)) if all_pos.size else 1.0

    # ---------- B) Per-gap completion in-place ----------
    for n in range(1, max_n + 1):
        col = f"T{n}"
        if col not in res.columns:
            res[col] = 0.0
            tcols.append(col)

        times_n  = res[col].to_numpy(dtype=float)
        reached  = (N_obs >= n)
        cens_gap = (N_obs == n) & (ls_obs == 0)
        events_n = (reached & ~cens_gap).astype(int)

        # KM fit on those who reached n and have positive Tn
        fit_mask   = reached & (times_n > 0)
        fit_times  = times_n[fit_mask]
        fit_events = events_n[fit_mask]

        # Unconditional KM mean (if possible) for brand-new gaps
        km_mean_Tn = np.nan
        if (fit_times.size > 0) and (fit_events.sum() > 0):
            kmf = KaplanMeierFitter().fit(fit_times, event_observed=fit_events)
            t_km = kmf.survival_function_.index.values.astype(float)
            S_km = kmf.survival_function_["KM_estimate"].to_numpy()
            S_prev = np.concatenate(([1.0], S_km[:-1]))
            dF = S_prev - S_km
            km_mean_Tn = float(np.sum(t_km * dF)) if dF.size else np.nan

            # --- Impute censored-at-gap rows directly with the updated imputer ---
            # Build a KM dataframe on *all* reached-n rows with positive time:
            # include both event rows and censored-at-gap rows.
            km_df = pd.DataFrame({
                "time":  res.loc[fit_mask, col].to_numpy(),
                "last_state": fit_events
            })
            km_imputed = km_impute_times(km_df, time_col="time", status_col="last_state",
                                         integer_mode=False, show_log=False).to_numpy()
            # Write back only to those censored-at-gap rows (that are in fit_mask)
            fit_positions = np.where(fit_mask)[0]
            for pos_local, idx_global in enumerate(fit_positions):
                if cens_gap[idx_global]:
                    res.at[idx_global, col] = float(km_imputed[pos_local])

        # --- Create “brand-new” gaps for rows where N_hat >= n and current Tn <= 0 ---
        need_n  = (N_hat >= n)
        no_time = (res[col].to_numpy() <= 0)
        to_fill = np.where(need_n & no_time)[0]

        for i in to_fill:
            if not np.isnan(km_mean_Tn) and km_mean_Tn > 0:
                res.at[i, col] = km_mean_Tn
            else:
                # fallback: individual's mean of existing positive gaps (before n)
                prev_vals = []
                for k in range(1, n):
                    ck = f"T{k}"
                    if ck in res.columns:
                        vk = res.at[i, ck]
                        if pd.notna(vk) and vk > 0:
                            prev_vals.append(float(vk))
                if prev_vals:
                    res.at[i, col] = float(np.mean(prev_vals))
                else:
                    res.at[i, col] = global_median

    # Zero out gaps beyond N_hat (rectangular shape)
    for n in range(1, max_n + 1):
        col = f"T{n}"
        res.loc[N_hat < n, col] = 0.0

    # Finalize: set N = N_hat and last_state = 1 for all rows
    res["N"] = N_hat
    res["last_state"] = 1

    out_cols = [f"T{k}" for k in range(1, max_n + 1)] + ["N", "last_state"]
    res = res[out_cols].copy()

    if show_log:
        print(f"Output shape: {res.shape}. Returned T1..T{max_n}, N, last_state (all ones).")

    return res

#### 0.2.3. Empirical functions

In [ ]:
def empirical_r_moments(dataframe, k=1, method="naive"):
    """
    Empirical moment E[r^k],   r = # recurrences before death.

    Parameters
    ----------
    dataframe : DataFrame with columns N and last_state
    k         : integer moment order (k ≥ 1)
    method    : "naive" or "km"

    Returns
    -------
    float
    """
    r = dataframe["N"].to_numpy() - 1
    events = (dataframe["last_state"] != 0).astype(int).to_numpy()  # 1 = death

    # ------------ naive: deaths only ---------------------------
    if method.lower() == "naive":
        obs = r[events == 1]
        return np.mean(obs ** k) if obs.size else np.nan

    # ------------ Kaplan–Meier for the discrete count ----------
    kmf = KaplanMeierFitter().fit(r, event_observed=events)

    j = np.arange(0, r.max() + 1)                     # 0,1,2,…,r.max()
    S = kmf.predict(j + 1e-300).values                # P(r > j)

    delta = j ** (k - 1)
    return float(k * np.sum(delta * S))

def empirical_conditional_times_moments(dataframe, n, l, k=1, equality=False, method="naive"):
    """
    Sample k‑th moment of T_n | r≥l  (or r==n‑1 when equality=True).

    Parameters
    ----------
    dataframe : DataFrame   with T1,…,TN, N, last_state.
    n         : int         index of the inter-event time T_n
    l         : int         lower bound on r (number of recurrences before death).
    k         : int         moment order (k ≥ 1).
    equality  : bool        if True use r == n‑1 instead of r ≥ l.
    method    : str         "naive" = drop censored, "km" = Kaplan–Meier adjustment.

    Returns
    -------
    float  estimated E[T_n^k | condition].
    """
    if l < n - 1:
        raise ValueError("l must be at least n‑1.")

    # ------------------------------------------------------------------
    # 1. Restrict on the r‑condition
    # ------------------------------------------------------------------
    dataframe = dataframe[dataframe["N"] - 1 == n - 1] if equality else dataframe[dataframe["N"] - 1 >= l]
    
    if dataframe.empty:
        return np.nan

    times = dataframe[f"T{n}"].values
    events = (dataframe["last_state"] != 0).values   # 0 = censorship

    # ------------------------------------------------------------------
    # 2a. Kaplan–Meier moment  (includes censored observations)
    # ------------------------------------------------------------------
    if method.lower() == "km":
        kmf = KaplanMeierFitter().fit(times, event_observed=events)
        t = kmf.survival_function_.index.values
        S = kmf.survival_function_['KM_estimate'].values

        # Escape if nobody experienced the event
        if len(t) == 0:
            return np.inf

        t_ext = np.concatenate([t, [t[-1]]])
        incr  = np.concatenate([[1.0], S]) - np.concatenate([S, [0.0]])
        return np.sum((t_ext ** k) * incr)

    # ------------------------------------------------------------------
    # 2b. Naive moment  (drops right‑censored gaps)
    # ------------------------------------------------------------------
    censorship = (dataframe["N"] == n) & (dataframe["last_state"] == 0)
    observed_times = dataframe.loc[censorship == False, f"T{n}"].values

    if observed_times.size == 0:
        return np.nan

    return np.mean(observed_times ** k)

def empirical_conditional_times_medians(dataframe, n, l, equality=False, method="naive"):
    """
    Median of T_n | r≥l     (or r==n‑1 when equality=True).

    Parameters
    ----------
    dataframe : DataFrame with T1,…,TN, N, last_state.
    n, l      : as in the moments routine.
    equality  : if True use r == n‑1 instead of r ≥ l.
    method    : "naive"  – take the sample median of observed events only
                "km"     – Kaplan–Meier 50‑th percentile (includes censored).

    Returns
    -------
    float      median, np.nan if no usable observations,
               np.inf  if KM curve never drops below 0.5.
    """
    if l < n - 1:
        raise ValueError("l must be at least n‑1.")

    # ------------------------------------------------------------------
    # 1. Restrict on the r‑condition
    # ------------------------------------------------------------------
    dataframe = dataframe[dataframe["N"] - 1 == n - 1] if equality else dataframe[dataframe["N"] - 1 >= l]
    
    if dataframe.empty:
        return np.nan

    times = dataframe[f"T{n}"].values
    events = (dataframe["last_state"] != 0).values   # 0 = censorship

    # ------------------------------------------------------------------
    # 2a. Kaplan–Meier median  (includes censored observations)
    # ------------------------------------------------------------------
    if method.lower() == "km":
        kmf = KaplanMeierFitter().fit(times, event_observed=events)
        try:
            return float(kmf.median_survival_time_)
        except ValueError:                 # survival never reached 0.5
            return np.inf

    # ------------------------------------------------------------------
    # 2b. Naive median  (drops right‑censored gaps)
    # ------------------------------------------------------------------
    censorship = (dataframe["N"] == n) & (dataframe["last_state"] == 0)
    observed_times = dataframe.loc[censorship == False, f"T{n}"].values

    if observed_times.size == 0:
        return np.nan
    
    return np.median(observed_times)

def empirical_conditional_times_correlations(dataframe, n, p, l, equality=False, method="naive"):
    """
    Sample correlation  ρ(T_n , T_p  |  condition)

    Parameters
    ----------
    dataframe : wide-format DataFrame with T1 … TN, N, last_state
    n, p      : gap indices  (1-based).  They may be given in any order.
    l         : lower bound on   r = # recurrences  (see paper)
    equality  : if True use  r == max(n-1, p-1)  instead of r ≥ l
    method    : "naive"       – drop records censored at *either* gap
                "km"   – impute T_p with the KM half-life rule

    Returns
    -------
    float  (np.nan if < 2 usable observations)
    """
    
    ###################################################################################################
    #################################### AUXILIARY FUNCTIONS ##########################################
    ###################################################################################################
    def _km_half_life_impute(times: np.ndarray, events: np.ndarray) -> np.ndarray:
        """
        “Half-life” Kaplan–Meier imputation used by the old R helper
        (kaplan_meier_imputation):

            – fit KM on (times, events)
            – for every censored observation replace its gap length t₀ by
            the *first* time t where   Ŝ(t) ≤ ½·Ŝ(t₀–)

        Parameters
        ----------
        times   : 1-D array of gap lengths (float)
        events  : 1-D 0/1 array (1 = event, 0 = right-censored at `times[i]`)

        Returns
        -------
        1-D array of the same length with censored gaps imputed.
        """
        kmf = KaplanMeierFitter().fit(times, event_observed=events)

        t_km   = kmf.survival_function_.index.values          # sorted unique times
        S_km   = kmf.survival_function_['KM_estimate'].values
        S_prev = np.concatenate(([1.0], S_km[:-1]))           # Ŝ(t-)

        out = times.copy().astype(float)
        cens_idx = np.where(events == 0)[0]

        for i in cens_idx:
            t0   = times[i]
            # survival just *before* t0
            S0   = S_prev[np.searchsorted(t_km, t0, side='left')]
            target = 0.5 * S0
            below = np.where(S_km <= target)[0]
            if below.size:           # survival eventually falls below ½·S0
                out[i] = t_km[below[0]]
            else:                    # never reaches ½·S0  →  use last KM time
                out[i] = t_km[-1]
        return out
    ###################################################################################################
    ################################### END OF AUXILIARY FUNCTIONS ####################################
    ###################################################################################################
    
    # -------------------- sanity checks ---------------------------
    if l < max(n - 1, p - 1):
        raise ValueError("l must be ≥ max(n-1 , p-1)")
    if n == p:
        raise ValueError("n and p must be different gaps")

    n, p = int(n), int(p)

    # -------------------- 1.  r-condition -------------------------
    cond = (dataframe["N"] - 1 == max(n - 1, p - 1)) if equality else (dataframe["N"] - 1 >= l)
    sub = dataframe[cond].copy()
    if sub.empty:
        return np.nan

    # keep only rows where *both* gaps are recorded (N ≥ p)
    sub = sub[sub["N"] >= p]
    if sub.empty:
        return np.nan

    # shorthand arrays ------------------------------------------------
    Tn = sub[f"T{n}"].to_numpy(dtype=float)
    Tp = sub[f"T{p}"].to_numpy(dtype=float)

    # -------- 2a.  KM-imputed version -------------------------------
    if method.lower() == "km":
        # “event” = gap p ends in death OR another recurrence
        events_p = ((sub["N"] > p) | (sub["last_state"] != 0)).astype(int).to_numpy()
        
        Tp = _km_half_life_impute(Tp, events_p)

        # rows where gap n or p is missing (0) are dropped
        mask = (Tn > 0) & (Tp > 0)

    # -------- 2b.  Naive version ------------------------------------
    else:       # method == "naive"
        # event at n ⇔   not censored right after gap n
        ev_n = ~((sub["N"] == n) & (sub["last_state"] == 0))
        ev_p = ~((sub["N"] == p) & (sub["last_state"] == 0))
        mask = ev_n & ev_p & (Tn > 0) & (Tp > 0)

    if mask.sum() < 2:
        return np.nan

    r, _ = pearsonr(Tn[mask], Tp[mask])
    return float(r)

#### 0.2.4. Theoretical functions

In [ ]:
def theoretical_r_moments(alpha0, D0, D1, k):
    """
    Given CS-MAP parameters, computes the moment of order k of the number of recurrences r, i.e. E(r^k).
    
    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - k: order of the moment
    
    Returns:
    - moment: moment of order k of r, E(r^k)
    """
    m = len(alpha0)
    
    # Compute the base matrix, P*I_R(I-P*I_R)^(-1)
    P_star = - np.linalg.inv(D0) @ D1
    P_star_reduced = P_star.copy()
    P_star_reduced[:, -1] = 0
    resolvent = np.linalg.inv(np.eye(m) - P_star_reduced)
    M = P_star_reduced @ resolvent

    # Compute powers of the base matrix
    M_powers = [np.eye(m)]
    for j in range(1, k + 1):
        M_powers.append(M_powers[j-1] @ M)

    # Sum the powers of the helper matrix and compute the moment
    moment_matrix = np.zeros((m, m))
    for j in range(1, k + 1):
        moment_matrix += factorial(j) * stirling2(k, j, exact=True) * M_powers[j]
    moment = np.sum(alpha0 @ moment_matrix)

    return moment

# --------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------
def _matrix_power(A, k):
    """Fast integer power (binary-squaring, O(log k))."""
    result = np.eye(A.shape[0])
    base   = A.copy()
    exp    = k
    while exp:
        if exp & 1:
            result = result @ base
        base = base @ base
        exp >>= 1
    return result


# --------------------------------------------------------------------
# 95-th percentile of r
# --------------------------------------------------------------------
def theoretical_r_percentile(alpha0, D0, D1, q=0.5, tol=1e-12,  max_n=1_000_000):
    """
    Return the *smallest integer n* such that
        1 – α₀ (P* I_R)^{n+1} 1  ≥  q   (default q=0.95)

    Parameters
    ----------
    alpha0 : 1-D array (m,)
    D0, D1 : (m,m) CS-MAP matrices
    q      : quantile in (0,1)
    tol    : numerical tolerance for the CDF comparison
    max_n  : hard cap to avoid infinite loops if the chain is ill-posed
    """
    m = len(alpha0)

    # ----- build  P*I_R  -------------------------------------------
    P_star = -np.linalg.inv(D0) @ D1              # embedded DTMC
    PIR    = P_star.copy()
    PIR[:, -1] = 0.0                              # zero transitions to the exit state
    PIR    = PIR[:-1, :-1]                        # keep the (m-1)×(m-1) recurrence block

    # initial distribution restricted to the recurrence set
    alphaR = alpha0[:-1] / alpha0[:-1].sum()
    ones   = np.ones(m-1)

    # quick lambda to compute CDF(n) = 1 – α (PIR)^{n+1} 1
    def cdf(n: int) -> float:
        tail = (alphaR @ _matrix_power(PIR, n+1) @ ones).item()
        return 1.0 - tail

    # guard against an ill-posed system
    if cdf(max_n) + tol < q:
        raise RuntimeError("max_n too small or the chain hardly exits; "
                           "increase `max_n` or re-check the parameters.")

    # ----- exponential search for an upper bound where CDF ≥ q ----
    n_hi = 0
    while cdf(n_hi) + tol < q:
        n_hi = 2 * n_hi + 1
        if n_hi > max_n:
            n_hi = max_n
            break
    n_lo = max(0, (n_hi - 1) // 2)

    # ----- integer bisection --------------------------------------
    while n_lo + 1 < n_hi:
        mid = (n_lo + n_hi) // 2
        if cdf(mid) + tol >= q:
            n_hi = mid
        else:
            n_lo = mid

    return n_hi

def theoretical_times_moments(alpha0, D0, D1, n, l, k=1, equality=False):
    """
    Given CS-MAP parameters, computes the k-th order moment of the n-th inter-event time T_n, conditioned to r>=l, i.e. E(T_n^k | r>=l). The condition r=n-1 may also be input.
    
    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - n: index of the inter-event time T_n
    - l: minimum number of recurrences after n. You must enter l>=n-1 (r=n-1 implies that T_n is a death time)
    - k: order of moment
    - equality: boolean that imposes the r==n-1 condition if True, and the r>=l condition if false

    Returns:
    - moment: scalar, E(T_n^k | r >= l)
    """
    # Calculate the inverse of D0
    inv_D0 = np.linalg.inv(D0)

    # Calculate the reduced transition matrix
    P_star = - inv_D0 @ D1
    P_star_recurrence = P_star.copy()
    P_star_recurrence[:, -1] = 0

    # Decide whether to calculate with the condition r>=l or r==n-1
    if equality==False:
        # Calculate the moment
        numerator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ np.linalg.matrix_power(inv_D0, k) @ np.linalg.matrix_power(P_star_recurrence, l - (n - 1)))
        denominator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, l))
    else:
        # Construct the other reduced transition matrix
        m = len(alpha0)
        P_star_death = np.zeros((m, m))
        P_star_death[:, -1] = P_star[:, -1]

        # Calculate the moment
        numerator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ np.linalg.matrix_power(inv_D0, k) @ P_star_death)
        denominator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ P_star_death)

    moment_no_sign = factorial(k) * numerator / denominator
    moment = - moment_no_sign if k%2!=0 else moment_no_sign

    return moment

def theoretical_times_correlations(alpha0, D0, D1, n, p, l, equality=False):
    """
    Given CS-MAP parameters, computes the correlation of the n-th and p-th inter-event times T_n, T_p conditioned to r>=l, i.e. rho(T_n, T_p | r>=l). The condition r=n-1 may also be input.

    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - n: first index of the inter-event times
    - p: second index of the inter-event times (this is the bigger one)
    - l: minimum number of recurrences after max(n, p). You must enter l>=max(n-1, p-1) (r=max(n-1, p-1) implies that T_max(n-1, p-1) is a death time)
    - equality: boolean that imposes the r==max(n-1, p-1) condition if True, and the r>=l condition if False.

    Returns:
    - moment: scalar, rho(T_n, T_p | r >= l)
    """
    # Calculate the inverse of D0
    inv_D0 = np.linalg.inv(D0)
    
    # Calculate the reduced transition matrix
    P_star = - inv_D0 @ D1
    P_star_recurrence = P_star.copy()
    P_star_recurrence[:, -1] = 0
    
    # Order the indices
    if n > p:
        n, p = p, n

    # Calculates the core elements of the correlation formula
    if equality==False:
        # Calculate the moments
        ETT = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, p - n) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, l - (p - 1))) / np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, l))
        ET1 = theoretical_times_moments(alpha0, D0, D1, n, l, 1, equality)
        ET2 = theoretical_times_moments(alpha0, D0, D1, p, l, 1, equality)
        ET1_sq = theoretical_times_moments(alpha0, D0, D1, n, l, 2, equality)
        ET2_sq = theoretical_times_moments(alpha0, D0, D1, p, l, 2, equality)
    else:
        # Construct the other reduced transition matrix
        m = len(alpha0)
        P_star_death = np.zeros((m, m))
        P_star_death[:, -1] = P_star[:, -1]

        # Calculate the moments
        ETT = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, p - n) @ inv_D0 @ P_star_death) / np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, p - 1) @ P_star_death)
        ET1 = theoretical_times_moments(alpha0, D0, D1, n, l, 1, equality)
        ET2 = theoretical_times_moments(alpha0, D0, D1, p, l, 1, equality)
        ET1_sq = theoretical_times_moments(alpha0, D0, D1, n, l, 2, equality)
        ET2_sq = theoretical_times_moments(alpha0, D0, D1, p, l, 2, equality)

    # Compute correlation coefficient
    cov = ETT - ET1 * ET2
    std_ET1 = np.sqrt(ET1_sq - ET1 ** 2)
    std_ET2 = np.sqrt(ET2_sq - ET2 ** 2)
    correlation = cov / (std_ET1 * std_ET2)

    return correlation

#### 0.2.5. Optimizer class

In [ ]:
def loglikelihood(dataframe, alpha0, D0, D1, progress=False, eps=1e-300):
    """
    Log-likelihood of a CS-MAP model on possibly censored (or complete) data, programmed with parallelization techniques.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Must include columns T1, T2, ..., TN, N (num gaps), and last_state:
        - 0 = censored
        - 1 = death

    alpha0 : array of shape (m,)
        Initial state distribution.
    D0, D1 : arrays of shape (m, m)
        Generator matrices of the CS-MAP.
    progress : bool
        If True, prints progress every ~10% of rows.
    eps : float
        Logarithmic safeguard to avoid log(0).

    Returns
    -------
    float
        Total log-likelihood over all paths in the DataFrame.
    """
    m = D0.shape[0]

    # ------------------------------------------------------------------
    # 1.  try a single spectral decomposition of D0
    # ------------------------------------------------------------------
    try:
        # eigen-decomposition  D0 = C Λ C⁻¹
        eigvals, C = eig(D0)
        if np.linalg.matrix_rank(C) < m:
            raise LinAlgError("Diagonalization failed: numerical error")
        C_inv = inv(C)

        # pre-allocate the exponentials of Λ multiplied by t on the fly
        @lru_cache(maxsize=512)
        def _expD0_cached(t_float):
            t = float(t_float)
            # diag( e^{λ t} ) – keep real part if the imaginary residue is tiny
            diag = np.diag(np.exp(eigvals * t))
            return np.real_if_close(C @ diag @ C_inv)
        
        def _expD0(t):
            return _expD0_cached(float(t))

    except LinAlgError:
        # Not diagonalizable numerically → robust fallback
        print("Diagonalization failed: D0 is not diagonalizable")
        def _expD0(t):
            return expm(D0 * t)

    # Obtain the reduced D1 matrices and the time columns from the dataset
    D1_recurrence = D1.copy()
    D1_recurrence[:, -1] = 0.0
    D1_death  = D1[:, [-1]]
    
    # Ensure T-columns are in order T1, T2, ...
    time_columns = sorted([c for c in dataframe.columns if c.startswith("T")],
                          key=lambda x: int(x[1:]))
    
    n_rows = len(dataframe)
    step = max(1, n_rows // 10)

    lock = Lock()
    processed = 0

    # ------------------------------------------------------------------
    # 2. Per-row worker (runs in parallel threads)
    # ------------------------------------------------------------------
    def _row_loglik(row):
        nonlocal processed
        N_i = int(row.N)
        if N_i <= 0:
            with lock:
                processed += 1
                if progress and (processed % step == 0 or processed == n_rows):
                    print(f"{processed}/{n_rows} rows processed")
            return -np.inf  # should not happen; keep safe

        # Extract times for this path (first N_i entries)
        row_times = [getattr(row, col) for col in time_columns[:N_i]]

        # Begin calculating the logarithm summand
        # Perform all matrix calculations except the last one
        v = alpha0.copy()
        for t in row_times[:-1]:
            v = v @ _expD0(t) @ D1_recurrence
        v = v @ _expD0(row_times[-1])

        # Perform the last matrix calculation if the row is not censored
        if int(row.last_state) == 1:
            v = v @ D1_death
        # last_state == 0  → censored, keep v
        v = float(v.sum())

        # Emergency exit code, should not happen unless overflow occurs
        if v < 0.0 or not np.isfinite(v):
            return np.log(eps)
        
        with lock:
            processed += 1
            if progress and (processed % step == 0 or processed == n_rows):
                print(f"{processed}/{n_rows} rows processed")

        # Calculate the logarithm of the final matrix product
        v = np.log(v + eps)
        return v

    # ------------------------------------------------------------------
    # 3. Parallel map over rows and reduce
    # ------------------------------------------------------------------
    total_loglikelihood = 0.0
    with ThreadPoolExecutor(max_workers=min(os.cpu_count() or 1, n_rows)) as ex:
        for contribution in ex.map(_row_loglik, dataframe.itertuples(index=False)):
            if not np.isfinite(contribution):
                return np.log(eps)
            total_loglikelihood += contribution

    return total_loglikelihood

def objective_function(alpha0, D0, D1, dataframe, R_hat, formula="pepa", cost_coefficient=0, eps=1e-100):
    """
    Scoring functions:
    - "pepa" or 1: F1 from Pepa's presentation.
    - "paper" or 2: F2 from Álvaro's paper.
    - "tfm" or 3: F3 from Álvaro's TFM.
    
    The "R" vectors (theoretical and empirical) calculated contain the following values:
        R1 = E[r]
        R2 = E[T1]
        R3 = E[T2 | r≥1]
        R4 = E[T3 | r≥2]
        R5 = ρ(T1,T2 | r≥1)
        R6 = ρ(T1,T3 | r≥2)
        R7 = ρ(T2,T3 | r≥2)
        
    Parameters:
    - alpha0: MAP initial probability vector.
    - D0: MAP invisible transitions matrix.
    - D1: MAP visible transitions matrix.
    - dataframe: CS-MAP dataframe, in the appropriate format (see the preprocessed "bladder1" and "colorectal" datasets).
    - R_hat: The empirical values of the dataset.
    - formula: Keyword that selects the desired objective function.
    - cost_coefficient: Penalty weight of the cost function.

    Returns:
    - float: value of the chosen scoring function
    """
    
    # Calculate theoretical conditional correlations of inter-event times (present in all objective functions)
    R = np.zeros(7, dtype=float)
    R[4] = theoretical_times_correlations(alpha0, D0, D1, 1, 2, l=1)
    R[5] = theoretical_times_correlations(alpha0, D0, D1, 1, 3, l=2)
    R[6] = theoretical_times_correlations(alpha0, D0, D1, 2, 3, l=2)
      
    # Scoring function F1 from Pepa's presentation
    if formula=="pepa" or formula==1:
        loglik_value = loglikelihood(dataframe, alpha0, D0, D1, progress=False, eps=eps)
        penalty = np.sum((R[4:7] - R_hat[4:7])**2)
        return loglik_value - cost_coefficient * penalty
    
    # Scoring function F2 from Álvaro's paper
    elif formula=="paper" or formula==2:
        loglik_value = loglikelihood(dataframe, alpha0, D0, D1, progress=False, eps=eps)
        penalty = np.sqrt(np.sum(
            ((R[4:7] - R_hat[4:7]) / (1 + np.abs(R_hat[4:7])))**2))
        return loglik_value * (1 + (cost_coefficient / 3) * penalty)
    
    # Scoring function F3 from Álvaro's TFM
    elif formula=="tfm" or formula==3:
        R[0] = theoretical_r_moments(alpha0, D0, D1, k=1)
        R[1] = theoretical_times_moments(alpha0, D0, D1, n=1, l=0, k=1)
        R[2] = theoretical_times_moments(alpha0, D0, D1, n=2, l=1, k=1)
        R[3] = theoretical_times_moments(alpha0, D0, D1, n=3, l=2, k=1)
        
        term1 = (R[0] - R_hat[0])**2 / 3
        term2 = np.sum((R[1:7] - R_hat[1:7])**2) / 9
        return np.sqrt(term1 + term2)
    
    else:
        raise ValueError("No valid function was specified.")

# ------------------------------------------------------------------
# 1.  pack   (alpha0, D0, D1)  --->  θ ∈ ℝ^(2m²−1)   --------------
# ------------------------------------------------------------------
def params_to_vec(alpha0, D0, D1):
    """
    α0      : (m,)
    D0, D1  : (m, m)

    returns : 1-D ndarray of length  (m−1) + (m²−m) + m²  =  2 m² − 1
              [ α(1)…α(m−1) |  D0 off-diag row-wise | D1 full row-wise ]
    """
    alpha0 = np.asarray(alpha0, dtype=float)
    D0     = np.asarray(D0,     dtype=float)
    D1     = np.asarray(D1,     dtype=float)
    m      = alpha0.size

    # -- α0  (skip last entry) --------------------------------------
    vec = [*alpha0[:m-1]]

    # -- D0  (off diagonals only) -----------------------------------
    for i in range(m):
        vec.extend(D0[i, :i])      # j < i
        vec.extend(D0[i, i+1:])    # j > i

    # -- D1  (all entries) ------------------------------------------
    vec.extend(D1.ravel())

    return np.asarray(vec, dtype=float)


# ------------------------------------------------------------------
# 2.  unpack θ  →  (alpha0, D0, D1)   ------------------------------
# ------------------------------------------------------------------
def vec_to_params(theta, m):
    """
    θ length = 2 m² – 1, laid out by `params_to_vec`:

        α(1)…α(m-1) | D0-off-diag row-wise | D1 (full)

    Returns
    -------
    alpha0, D0, D1  where           (D0 + D1)·1 = 0  and  diag(D0) < 0
    """
    theta = np.asarray(theta, float)
    if theta.size != 2*m*m - 1:
        raise ValueError("θ has the wrong length")

    pos = 0
    # ---------- α0 --------------------------------------------------
    alpha0 = np.empty(m)
    alpha0[:m-1] = theta[pos:pos+m-1]
    alpha0[m-1] = 1.0 - alpha0[:m-1].sum()
    alpha0 = abs(alpha0) / abs(alpha0).sum()
    pos += m-1
    #if (alpha0 < 0).any():
        #raise ValueError("α0 has a negative entry")

    # ---------- D0 off–diagonals -----------------------------------
    D0 = np.zeros((m, m))
    for i in range(m):
        left = i
        right = m - i - 1
        D0[i, :i] = theta[pos: pos+left];  pos += left
        D0[i, i+1:] = theta[pos: pos+right]; pos += right
    #  (pos is now  m-1 + m(m-1)  = m² – 1)
    #if (D0 < 0).any():
        #raise ValueError("D0 has a negative off-diagonal entry")

    # ---------- D1 full --------------------------------------------
    D1 = theta[pos:].reshape(m, m)
    #if (D1 < 0).any():
        #raise ValueError("D1 has a negative entry")

    # ---------- set D0 diagonals so that  (D0 + D1)·1 = 0 ----------
    for i in range(m):
        D0[i, i] = -(D0[i].sum() + D1[i].sum())
    return alpha0, D0, D1


# ----------------------------------------------------------------------
# 3.  negative objective function score (SciPy minimises)  -------------
# ----------------------------------------------------------------------
def _neg_objective(theta, dataframe, m, R_hat, formula, cost_coeff, eps):
    alpha0, D0, D1 = vec_to_params(theta, m)
    score = objective_function(alpha0, D0, D1,
                               dataframe,
                               R_hat,
                               formula=formula,
                               cost_coefficient=cost_coeff,
                               eps=eps)
    
    # If something blew up, give the optimiser a huge penalty
    if not np.isfinite(score):
        score = -1e300

    # maximise  ⇒  minimise the negative
    return -score
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 4.  bounds enforcing  ----------------------------
# ----------------------------------------------------------------------

def make_bounds(m, eps=1e-300):
    """
    Constructs bounds for the optimization problem.

    Parameters
    ----------
    m : int
        Model order.

    Returns
    -------
    bounds : scipy.optimize.Bounds
        Bounds object for minimize().
    """
    n_alpha = m - 1
    n_d0 = m * (m - 1)      # Off-diagonal D0
    n_d1 = m * m            # Full D1
    total = n_alpha + n_d0 + n_d1

    lb = np.zeros(total) + eps
    ub = np.full(total, np.inf)

    # alpha0 bounds: [0, 1 - eps]
    #ub[:n_alpha] = 1.0 # NOT NECESSARY: THE SUM CONSTRAINT MAKES THESE REDUNDANT. NOT ENFORCING THEM REQUIRES MORE INITIALIZATIONS, BUT PROVIDES BETTER RESULTS

    return Bounds(lb, ub, keep_feasible=True)


# ---------- 0. helper -------------------------------------------------
def make_simplex_constraint(m):
    """
    α₁,…,α_{m-1} are the first (m-1) entries of θ.
    Enforce   ∑ α_i  ≤  1 – eps     (upper bound only).
    """
    n_total  = 2*m*m - 1
    n_alpha  = m - 1

    A = np.zeros((1, n_total))
    A[0, :n_alpha] = 1.0           # sum of the first (m-1) variables

    # lower bound = -inf  (we already bound each α_i ≥ eps component-wise)
    lb = -np.inf
    ub = 1.0
    return LinearConstraint(A, lb, ub, keep_feasible=True)

# ----------------------------------------------------------------------
# 5.  driver -----------------------------------------------------------
# ----------------------------------------------------------------------
def CSMAP_moments_matching_optimization(dataset,
                                        order,
                                        formula="pepa",
                                        initialisations=10,
                                        distribution="uniform",
                                        parameter_A=2,
                                        parameter_B=0.5,
                                        cost_coefficient=0,
                                        method="naive",
                                        loglike_eps=1e-12,
                                        lower_bound_eps=1e-300,
                                        max_scipy_iter=10,
                                        scipy_tol=1e-8,
                                        seed=None,
                                        verbose=False,
                                        *,
                                        prescreen_mode="p95",   # "p95" or "topk"
                                        top_k=20):              # used when prescreen_mode="topk"
    """
    Prescreens random initialisations by initial objective score and only
    runs SciPy on a subset:

      - prescreen_mode="p95"  → keep inits >= 95th percentile (default)
      - prescreen_mode="topk" → keep the top `top_k` inits by score

    Guarantees at least one candidate if prescreen would be empty.
    Returns (best_params, best_score, best_res).
    """
    rng = default_rng(seed)

    # --- build empirical target vector R_hat once ---
    R_hat = np.zeros(7, dtype=float)
    R_hat[4] = empirical_conditional_times_correlations(dataset, 1, 2, l=1, method=method)
    R_hat[5] = empirical_conditional_times_correlations(dataset, 1, 3, l=2, method=method)
    R_hat[6] = empirical_conditional_times_correlations(dataset, 2, 3, l=2, method=method)
    if formula in ("pepa", "paper"):
        R_hat[0] = float(dataset["N"].mean())
        R_hat[1] = empirical_conditional_times_moments(dataset, 1, l=0, k=1, method=method)
        R_hat[2] = empirical_conditional_times_moments(dataset, 2, l=1, k=1, method=method)
        R_hat[3] = empirical_conditional_times_moments(dataset, 3, l=2, k=1, method=method)

    # ---------- 1) Generate all random starts & score them ----------
    seeds = rng.integers(0, 2**32 - 1, size=initialisations, dtype=np.uint32)
    start_params_list = []
    init_scores = np.full(initialisations, -np.inf, dtype=float)

    for i in range(initialisations):
        pars = draw_CSMAP_parameters(order,
                                     distribution,
                                     parameter_A=parameter_A,
                                     parameter_B=parameter_B,
                                     random_seed=int(seeds[i]))
        start_params_list.append(pars)

        s = objective_function(pars["alpha0"], pars["D0"], pars["D1"],
                               dataframe=dataset, R_hat=R_hat,
                               formula=formula,
                               cost_coefficient=cost_coefficient,
                               eps=loglike_eps)
        if not np.isfinite(s):
            s = -np.inf
        init_scores[i] = s
        
        if verbose and i % int((initialisations/100)) == 0:
            print(f"{int((i/initialisations)*100)}% initialisations done")

    if verbose:
        finite_cnt = int(np.isfinite(init_scores).sum())
        print(f"[Prescreen] Scored {initialisations} starts ({finite_cnt} finite).")

    # ---------- 2) Select by mode ----------
    if prescreen_mode == "topk":
        k = max(1, min(int(top_k), initialisations))
        selected = np.argsort(init_scores)[-k:]                       # top-k by value
        selected = selected[np.argsort(init_scores[selected])[::-1]]  # descending
        if verbose:
            print(f"[Prescreen] Mode=topk  → keeping top {k}/{initialisations}.")
    else:
        # default "p95"
        finite_scores = init_scores[np.isfinite(init_scores)]
        cutoff = float(np.percentile(finite_scores, 95)) if finite_scores.size else -np.inf
        selected = np.where(init_scores >= cutoff)[0]
        if selected.size == 0:
            # fallback to top ceil(5%)
            k = max(1, int(np.ceil(0.05 * initialisations)))
            selected = np.argsort(init_scores)[-k:]
        selected = selected[np.argsort(init_scores[selected])[::-1]]
        if verbose:
            kept = selected.size
            print(f"[Prescreen] Mode=p95  → keeping {kept}/{initialisations} (cutoff={cutoff:.6g}).")

    # ---------- 3) Run SciPy only on the selected starts ----------
    best_score  = -np.inf
    best_theta  = None
    best_res    = None

    bounds = make_bounds(order, eps=lower_bound_eps)
    constraint = make_simplex_constraint(order)

    for idx, run in enumerate(selected, start=1):
        pars = start_params_list[run]
        theta0 = params_to_vec(pars["alpha0"], pars["D0"], pars["D1"])

        if verbose:
            print(f"\n[{idx}/{selected.size}] Optimising start #{run+1} "
                  f"(init score = {init_scores[run]:.6g})")

        res = minimize(_neg_objective,
                       theta0,
                       args=(dataset, order, R_hat, formula, cost_coefficient, loglike_eps),
                       method="trust-constr",
                       bounds=bounds,
                       constraints=[constraint],
                       options={"maxiter": max_scipy_iter,
                                "gtol": scipy_tol,
                                "verbose": 3 if verbose else 0})

        score = -res.fun  # back to “maximise” sense

        if score > best_score:
            best_score = score
            best_theta = res.x.copy()
            best_res   = res
            if verbose:
                print(f"  NEW BEST  score = {best_score:.6g}")
        elif verbose:
            print(f"  score = {score:.6g} (no improvement)")
        
        # Pretty-print the last iteration
        alpha0, D0, D1 = vec_to_params(res.x.copy(), order)
        if verbose:
            print("  alpha0 =", alpha0)
            print("  D0 =\n", D0)
            print("  D1 =\n", D1)
            print("  E(T1 | r>=0) =", theoretical_times_moments(alpha0, D0, D1, 1, 0, 1))
            print("  E(T2 | r>=1) =", theoretical_times_moments(alpha0, D0, D1, 2, 1, 1))
            print("  E(T3 | r>=2) =", theoretical_times_moments(alpha0, D0, D1, 3, 2, 1))
            print("  rho(T1, T2 | r>=1) =", theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
            print("  rho(T1, T3 | r>=2) =", theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
            print("  rho(T2, T3 | r>=2) =", theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))

    # ---------- 4) Unpack winner & return ----------
    if best_theta is None:
        # Edge guard: all optimisations failed → return best init
        fallback_idx = int(np.argmax(init_scores))
        if verbose:
            print("[Prescreen] No successful optimisation; returning best init.")
        alpha0, D0, D1 = (start_params_list[fallback_idx][k] for k in ("alpha0","D0","D1"))
        best_score = init_scores[fallback_idx]
    else:
        alpha0, D0, D1 = vec_to_params(best_theta, order)

    if verbose:
        print("\nFINAL BEST PARAMETERS:")
        print("  alpha0 =", alpha0)
        print("  D0 =\n", D0)
        print("  D1 =\n", D1)
        print("  best score =", best_score)

    return {"alpha0": alpha0, "D0": D0, "D1": D1}, best_score, best_res



class CSMAPOptimizer:
    def __init__(self, dataset, order=3, formula="pepa", initialisations=20, distribution="uniform", parameter_A=1, parameter_B=0, cost_coefficient=0, method="naive", lower_bound_eps=1e-300, loglike_eps=1e-16, max_scipy_iter=20, seed=RANDOM_SEED, verbose=False, prescreen_mode="p95", top_k=20):
        """
        Initialize optimizer with dataset and parameters. Runs optimization immediately.
        """
        self.dataset = dataset
        self.order = order
        self.formula = formula
        self.initialisations = initialisations
        self.distribution = distribution
        self.parameter_A = parameter_A
        self.parameter_B = parameter_B
        self.cost_coefficient = cost_coefficient
        self.method = method
        self.lower_bound_eps = lower_bound_eps
        self.loglike_eps = loglike_eps
        self.max_scipy_iter = max_scipy_iter
        self.seed = seed
        self.verbose = verbose
        self.prescreen_mode = prescreen_mode
        self.top_k = top_k

        # Run optimization upon creation
        self.best_params, self.best_score, self.best_res = CSMAP_moments_matching_optimization(
            dataset=self.dataset,
            order=self.order,
            formula=self.formula,
            initialisations=self.initialisations,
            distribution=self.distribution,
            parameter_A=self.parameter_A,
            parameter_B=self.parameter_B,
            cost_coefficient=self.cost_coefficient,
            method=self.method,
            lower_bound_eps=self.lower_bound_eps,
            loglike_eps=self.loglike_eps,
            max_scipy_iter=self.max_scipy_iter,
            seed=self.seed,
            verbose=self.verbose,
            prescreen_mode=self.prescreen_mode,
            top_k=self.top_k 
        )

    def print_summary(self, original_params=None):
        """
        Print a full summary: original params (if provided), estimated params, theoretical and empirical metrics.
        """
        print(f"\n===== DIMENSION {self.order} ESTIMATION =====\n")

        # ---------------- Original Parameters ----------------
        if original_params:
            alpha0, D0, D1 = original_params["alpha0"], original_params["D0"], original_params["D1"]
            print("\nORIGINAL PARAMETERS:")
            print("Original parameters score: ", loglikelihood(self.dataset, alpha0, D0, D1))
            print("Original alpha0 =", alpha0)
            print("Original D0 =\n", D0)
            print("Original D1 =\n", D1)
            print("\tOriginal inter-event time means:")
            for i in range(1, 4):
                print(f"\t E(T_{i} | r>={i-1}): ", theoretical_times_moments(alpha0, D0, D1, i, i-1, 1))
            print("\tOriginal inter-event time correlations:")
            print("\t rho(T_1, T_2 | r>=1): ", theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
            print("\t rho(T_1, T_3 | r>=2): ", theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
            print("\t rho(T_2, T_3 | r>=2): ", theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))

        # ---------------- Estimated Parameters ----------------
        alpha0_hat, D0_hat, D1_hat = self.best_params["alpha0"], self.best_params["D0"], self.best_params["D1"]

        print("\nBEST ESTIMATED PARAMETERS:")
        print("Best estimated score:", self.best_score)
        print("Best estimated alpha0 =", alpha0_hat)
        print("Best estimated D0 =\n", D0_hat)
        print("Best estimated D1 =\n", D1_hat)
        print("\tBest estimated inter-event time means:")
        for i in range(1, 4):
            print(f"\t \\hat(E(T_{i} | r>={i-1})): ", theoretical_times_moments(alpha0_hat, D0_hat, D1_hat, i, i-1, 1))
        print("\tBest estimated inter-event time correlations:")
        print("\t \\hat(rho(T_1, T_2 | r>=1)): ", theoretical_times_correlations(alpha0_hat, D0_hat, D1_hat, 1, 2, 1))
        print("\t \\hat(rho(T_1, T_3 | r>=2)): ", theoretical_times_correlations(alpha0_hat, D0_hat, D1_hat, 1, 3, 2))
        print("\t \\hat(rho(T_2, T_3 | r>=2)): ", theoretical_times_correlations(alpha0_hat, D0_hat, D1_hat, 2, 3, 2))
        
        # ---------------- Empirical Results ----------------
        print("\nEMPIRICAL RESULTS:")
        print("\tEmpirical inter-event time means:")
        for i in range(1, 4):
            print(f"\t \\bar(T_{i} | r>={i-1}): ", empirical_conditional_times_moments(self.dataset, i, i-1, 1))

        print("\tEmpirical inter-event time correlations:")
        print("\t r(T_1, T_2 | r>=1): ", empirical_conditional_times_correlations(self.dataset, 1, 2, 1))
        print("\t r(T_1, T_3 | r>=2): ", empirical_conditional_times_correlations(self.dataset, 1, 3, 2))
        print("\t r(T_2, T_3 | r>=2): ", empirical_conditional_times_correlations(self.dataset, 2, 3, 2))

### 0.3. Preprocess datasets

#### 0.3.1. bladder1

In [ ]:
def preprocess_bladder_wide(csv_path="bladder1.csv"):
    b = pd.read_csv(csv_path)
    for col in ["id", "enum", "start", "stop", "status"]:
        b[col] = pd.to_numeric(b[col], errors="coerce")

    ids = np.sort(b["id"].unique())
    n  = len(ids)
    p  = int(b["enum"].max())

    Tn = -np.ones((n, p), float)
    N  = -np.ones(n, int)
    # R tri-state: 0=censored, 1=recurrence, 2=death (keep for masking)
    last_state_r = -np.ones(n, int)
    id2row = {pid:i for i, pid in enumerate(ids)}

    for pid, g in b.groupby("id"):
        i = id2row[pid]
        g = g.sort_values("enum")
        N[i] = int(g["enum"].max())
        final_status = int(g.loc[g["enum"] == N[i], "status"].iloc[0])
        last_state_r[i] = min(final_status, 2)
        gaps = (g["stop"] - g["start"]).to_numpy()
        enums = g["enum"].astype(int).to_numpy()
        for j, gap in zip(enums, gaps):
            Tn[i, j-1] = float(gap)

    # --- R behavior baked in: if censored at boundary, remove T_N by setting it to -1
    mask = (N > 0) & (last_state_r == 0)
    rows = np.where(mask)[0]
    cols = (N[mask] - 1).astype(int)
    Tn[rows, cols] = -1.0

    # Your functions treat last_state==1 as "died" when censorship=False.
    # We'll keep that convention (though you’ll call censorship=True).
    last_state = (last_state_r == 2).astype(int)

    wide = pd.DataFrame({"id": ids})
    for j in range(1, p+1):
        wide[f"T{j}"] = Tn[:, j-1]
    wide["N"] = N
    wide["last_state"] = last_state        # 1=death, 0=not-death
    # (Optional) keep the original tri-state for audit/debug:
    wide["last_state_r"] = last_state_r    # 0=cens,1=recur,2=death
    return wide

bladder1 = preprocess_bladder_wide("bladder1.csv")

#### 0.3.2. colorectal

In [ ]:
def preprocess_colorectal_wide(csv_path="colorectal.csv"):
    c = pd.read_csv(csv_path)
    for col in ["id", "time1", "gap.time", "state"]:
        c[col] = pd.to_numeric(c[col], errors="coerce")

    ids = np.sort(c["id"].unique())
    n = len(ids)
    p = int(c.groupby("id").size().max())

    Tn = -np.ones((n, p), float)
    N  = -np.ones(n, int)
    # R uses last_state = state*2 (0=censored, 2=death)
    last_state_r = -np.ones(n, int)
    id2row = {pid:i for i, pid in enumerate(ids)}

    for pid, g in c.groupby("id"):
        i = id2row[pid]
        g = g.sort_values("time1")
        Ni = len(g)
        N[i] = Ni
        Tn[i, :Ni] = g["gap.time"].to_numpy()
        last_state_r[i] = int(g["state"].iloc[-1]) * 2  # 0 or 2

    # --- R behavior baked in: if censored at boundary, remove T_N
    mask = (N > 0) & (last_state_r == 0)
    rows = np.where(mask)[0]
    cols = (N[mask] - 1).astype(int)
    Tn[rows, cols] = -1.0

    # Keep your convention: last_state==1 means death (only used if censorship=False).
    last_state = (last_state_r == 2).astype(int)

    wide = pd.DataFrame({"id": ids})
    for j in range(1, p+1):
        wide[f"T{j}"] = Tn[:, j-1]
    wide["N"] = N
    wide["last_state"] = last_state        # 1=death, 0=censored
    wide["last_state_r"] = last_state_r    # 0 or 2
    return wide

colorectal = preprocess_colorectal_wide("colorectal.csv")

In [ ]:
def quick_eda(df: pd.DataFrame, name: str):
    total = len(df)
    censored = int((df["last_state"] == 0).sum())
    print(f"{name}:")
    print(f"  total patients                               = {total}")
    print(f"  censored patients                            = {censored} ({censored/total:.1%})")

    # highest r
    r_vals = pd.to_numeric(df["N"], errors="coerce").fillna(0).astype(int) - 1
    r_max = int(r_vals.max()) if len(r_vals) else 0
    print(f"  highest r (= max(N-1))                       = {r_max}")

    # ------------ Treat r as “one more time” ------------
    # Event for r ⇔ death observed (last_state != 0); censor ⇔ last_state == 0
    ls = pd.to_numeric(df["last_state"], errors="coerce").fillna(0).astype(int).to_numpy()
    ev_r = (ls != 0)
    cens_r = (ls == 0)

    r_array = r_vals.to_numpy()
    ev_r_vals = r_array[ev_r]
    avg_r = float(np.mean(r_array)) if r_array.size else np.nan
    denom_r = total

    if ev_r_vals.size == 0:
        max_r_str = "n/a"
        cens_r_str = "n/a"
    else:
        max_r_event = int(ev_r_vals.max())
        cens_ge_r = int((cens_r & (r_array >= max_r_event)).sum())
        prop_r = cens_ge_r / denom_r if denom_r > 0 else np.nan
        max_r_str = f"{max_r_event:d}"
        cens_r_str = f"{cens_ge_r}/{denom_r} ({prop_r:.1%})"

    avg_r_str = f"{avg_r:.6g}" if np.isfinite(avg_r) else "nan"
    print(f"  r:  max event time = {max_r_str} | cens ≥ max = {cens_r_str} | avg time = {avg_r_str}")

    # ------------ Per-gap triples for T1, T2, ... ------------
    tcols = sorted([c for c in df.columns if c.startswith("T") and c[1:].isdigit()],
                   key=lambda x: int(x[1:]))

    for c in tcols:
        n = int(c[1:])

        # keep only those who reached gap n
        reached_n = (df["N"] >= n)
        sub = df.loc[reached_n, :].copy()
        if sub.empty:
            continue

        times_n = pd.to_numeric(sub[c], errors="coerce").to_numpy()
        times_n = np.where(np.isfinite(times_n), times_n, 0.0)

        # define event/censor-at-gap
        Nn   = sub["N"].to_numpy()
        lso  = sub["last_state"].to_numpy()
        event_n = ((Nn > n) | ((Nn == n) & (lso != 0)))   # recurrence or terminal event at gap n
        cens_n  = ((Nn == n) & (lso == 0))

        # positive times only are meaningful
        pos = (times_n > 0)

        ev_times = times_n[event_n & pos]
        denom    = int(reached_n.sum())  # number who reached gap n
        avg_time = np.mean(times_n[pos]) if np.any(pos) else np.nan

        if ev_times.size == 0:
            max_event_str = "n/a"
            cens_str = "n/a"
        else:
            max_event_time = float(ev_times.max())
            cens_ge = int(((cens_n) & (times_n >= max_event_time) & pos).sum())
            prop    = cens_ge / denom if denom > 0 else np.nan
            max_event_str = f"{max_event_time:.6g}"
            cens_str = f"{cens_ge}/{denom} ({prop:.1%})"

        avg_str = f"{avg_time:.6g}" if np.isfinite(avg_time) else "nan"
        print(f"  {c}: max event time = {max_event_str} | cens ≥ max = {cens_str} | avg time = {avg_str}")

# usage
quick_eda(bladder1,  "bladder1_PREPROCESSED")
quick_eda(colorectal, "colorectal_PREPROCESSED")

quick_eda(complete_rd_csmap_dataset(bladder1), "bladder1_PREPROCESSED (imputed)")
quick_eda(complete_rd_csmap_dataset(colorectal), "colorectal_PREPROCESSED (imputed)")

### 0.4. Auxiliary code for plotting

In [ ]:
def _short_cost(c: float, rtol: float = 1e-12) -> str:
    """
    Labels exactly as:
      0, 100, 500, 1000, 5000, 10**4, 5*10**4, 10**5, 5*10**5, 10**6, 5*10**6, 10**7
    Fallback: integer string.
    """
    c = float(c)
    if abs(c) < 1e-12:
        return "0"

    # Explicit small values
    for v in (100.0, 500.0, 1000.0, 5000.0):
        if abs(c - v) <= rtol * max(1.0, v):
            return str(int(v))

    # Powers of 10 and 5*10^k for k >= 4
    for k in (4, 5, 6, 7):
        v1 = 10.0 ** k
        if abs(c - v1) <= rtol * v1:
            return f"10**{k}"
        v5 = 5.0 * v1
        if abs(c - v5) <= rtol * v5:
            return f"5*10**{k}"

    # Fallback: plain integer
    return str(int(round(c)))

def apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22):
    labels = [_short_cost(float(c)) for c in costs]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=rotation, ha="right", fontsize=fontsize)
    ax.figure.subplots_adjust(bottom=bottom)

## 1. EXPERIMENT 1 - Uncensored, simulated sample

### 1.1. Uncensored, simulated, order 3 sample

In [ ]:
'''alpha0 = np.array([0.24568483621151876, 0.73475600167806, 0.019559162110421344])
D0 = np.array([[-4.663690260637603, 0.4568485465011195, 0.22635780752402723],
                        [0.6576176569475198, -30.704738873051383, 13.999576643130636],
                        [6.769514005012695, 0.6694829770599626, -24.8059223179645]])
D1 = np.array([[2.6066541591363808, 0.30055562526772006, 1.0732741222083553],
                        [2.316807367368122, 13.364285820667178, 0.36645138493792945],
                        [3.001048257555971, 0.6276529446414862, 13.738224133694388]])'''
alpha0 = np.array([0.09811484, 0.78213864, 0.11974652])
D0 = np.array([[-0.00437278002, 0.000116544078,  0.000645668504],
                        [0.000136308408, -0.0934623725, 0.0000288499699],
                        [0.00181540856, 0.000937695154, -0.0137381432]])
D1 = np.array([[0.00286963027, 0.0000524964088, 0.000688440755],
                        [0.000972313710, 0.0564900897, 0.0358348106],
                        [0.00284716560, 0.00117900082, 0.00695887302]])

sample = simulate_csmap({"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=200, N_sim=4000, seed=RANDOM_SEED, progress=False)
print(loglikelihood(sample, alpha0, D0, D1))
print(empirical_conditional_times_moments(sample, 1, 0, 1))
print(empirical_conditional_times_moments(sample, 2, 1, 1))
print(empirical_conditional_times_moments(sample, 3, 2, 1))
print(empirical_conditional_times_correlations(sample, 1, 2, 1))
print(empirical_conditional_times_correlations(sample, 1, 3, 2))
print(empirical_conditional_times_correlations(sample, 2, 3, 2))
print(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1))
print(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1))
print(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))
print(empirical_r_moments(sample, 1))
print(theoretical_r_moments(alpha0, D0, D1, 1))

# Uncomment the following line if you want to generate the dataset as a .csv file within the computer
#sample.to_csv("uncensored_sample_order_3.csv")

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

In [ ]:
# DIMENSION 4 ESTIMATION

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=5*10**3,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 5 ESTIMATION

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=5,
    formula="pepa",
    initialisations=5*10**3,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

### 1.2. Uncensored, simulated, order 4 sample

In [ ]:
'''#HIGH CORRELATION PARAMETERS: rho12 = 0.1, rho23 = 0.2
alpha0 = np.array([0.23889327, 0.30089699, 0.1982954, 0.26191434])
D0 = np.array([
 [-1.23212367e+00, 1.34973080e-01, 1.68619209e-01, 1.07437198e-01],
 [ 3.28585892e-01, -1.37167458e+00, 1.29730792e-01, 7.61838809e-02],
 [ 1.42163531e-01, 8.04555564e-01, -1.94768870e+00, 6.94755509e-02],
 [ 3.53683935e-04, 3.67879532e-01, 1.06239354e+00, -3.34658980e+00]])
D1 = np.array([
 [2.77430375e-01, 6.80195221e-02, 1.28057757e-01, 3.47586526e-01],
 [2.95649147e-01, 8.10318151e-03, 5.02803460e-01, 3.06182297e-02],
 [1.50014820e-01, 3.40439605e-01, 4.40133920e-01, 9.05714028e-04],
 [1.46463982e-01, 1.59683674e-01, 2.97059809e-01, 1.31275558e+00]])'''

#HIGH CORRELATION PARAMETERS: rho12 = 0.2, rho13 = 0.13, rho23 = 0.22
alpha0 = np.array([0.21741426, 0.19636819, 0.44034113, 0.14587642])
D0 = np.array([
 [-2.02299244e+00, 1.14994768e-03, 1.74648420e-01, 1.06957334e-02],
 [3.22301221e-02, -2.14445825e-01, 1.25702213e-03, 2.16221675e-02],
 [3.68331420e-01, 4.70344664e-01, -2.94858276e+00, 1.54822923e-01],
 [1.08302259e-02, 1.23054362e+00, 5.65782422e-01, -3.10138088e+00]])
D1 = np.array([
 [1.25001803e+00, 1.34580942e-02, 3.88967348e-01, 1.84054874e-01],
 [1.70260226e-04, 1.19015780e-01, 3.86541810e-02, 1.49629138e-03],
 [1.37923540e+00, 5.42514060e-02, 1.09635356e-01, 4.11961594e-01],
 [5.89028973e-03, 2.82252340e-01, 1.55323308e-01, 8.50758673e-01]])

sample = simulate_csmap({"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=200, N_sim=4000, seed=RANDOM_SEED)
print(loglikelihood(sample, alpha0, D0, D1))
print(empirical_conditional_times_moments(sample, 1, 0, 1))
print(empirical_conditional_times_moments(sample, 2, 1, 1))
print(empirical_conditional_times_moments(sample, 3, 2, 1))
print(empirical_conditional_times_correlations(sample, 1, 2, 1))
print(empirical_conditional_times_correlations(sample, 1, 3, 2))
print(empirical_conditional_times_correlations(sample, 2, 3, 2))
print(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1))
print(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1))
print(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))
print(empirical_r_moments(sample, 1))
print(theoretical_r_moments(alpha0, D0, D1, 1))

# Uncomment the following line if you want to generate the dataset as a .csv file within the computer
#sample.to_csv("uncensored_sample_order_4.csv")

In [ ]:
# DIMENSION 3 ESTIMATION

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 0

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 100

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 500

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 1000

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 5000

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 10**4

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 5*10**4

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 10**5

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 5*10**5

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 10**6

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 5*10**6

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 4 ESTIMATION - cost coefficient = 10**7

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

In [ ]:
# DIMENSION 5 ESTIMATION

optimizer = CSMAPOptimizer(
    dataset=sample,
    order=5,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

## 2. EXPERIMENT 2 - Censored, simulated sample

In [ ]:
alpha0 = np.array([0.09811484, 0.78213864, 0.11974652])
D0 = np.array([[-0.00437278002, 0.000116544078,  0.000645668504],
                        [0.000136308408, -0.0934623725, 0.0000288499699],
                        [0.00181540856, 0.000937695154, -0.0137381432]])
D1 = np.array([[0.00286963027, 0.0000524964088, 0.000688440755],
                        [0.000972313710, 0.0564900897, 0.0358348106],
                        [0.00284716560, 0.00117900082, 0.00695887302]])

sample = simulate_csmap({"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=200, N_sim=200, seed=RANDOM_SEED, progress=False)
sample = apply_random_censoring(sample, how_many=0.75, seed=RANDOM_SEED)

print(loglikelihood(sample, alpha0, D0, D1))
print(loglikelihood(complete_rd_csmap_dataset(sample), alpha0, D0, D1))
print(empirical_conditional_times_moments(sample, 1, 0, 1, equality=False, method="naive"))
print(empirical_conditional_times_moments(sample, 2, 1, 1, equality=False, method="naive"))
print(empirical_conditional_times_moments(sample, 3, 2, 1, equality=False, method="naive"))
print(empirical_conditional_times_correlations(sample, 1, 2, 1, equality=False, method="naive"))
print(empirical_conditional_times_correlations(sample, 1, 3, 2, equality=False, method="naive"))
print(empirical_conditional_times_correlations(sample, 2, 3, 2, equality=False, method="naive"))
print(empirical_conditional_times_moments(sample, 1, 0, 1, equality=False, method="km"))
print(empirical_conditional_times_moments(sample, 2, 1, 1, equality=False, method="km"))
print(empirical_conditional_times_moments(sample, 3, 2, 1, equality=False, method="km"))
print(empirical_conditional_times_correlations(sample, 1, 2, 1, equality=False, method="km"))
print(empirical_conditional_times_correlations(sample, 1, 3, 2, equality=False, method="km"))
print(empirical_conditional_times_correlations(sample, 2, 3, 2, equality=False, method="km"))
print(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1))
print(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1))
print(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))
print(empirical_r_moments(sample, 1, method="naive"))
print(empirical_r_moments(sample, 1, method="km"))
print(theoretical_r_moments(alpha0, D0, D1, 1))

# Uncomment the following line if you want to generate the dataset as a .csv file within the computer
#sample.to_csv("censored_sample.csv")

In [ ]:
quick_eda(sample, "censored_testing_sample")

In [ ]:
ignore_censorship_sample = sample
impute_censorship_sample = complete_rd_csmap_dataset(sample)

### 2.1. Dropping censorship (work with the 25% of the sample)

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    method="naive",
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

### 2.2. Work with censorship loglikelihood

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    method="km",
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

### 2.3. Impute censorship

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_sample,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary(original_params={"alpha0": alpha0, "D0": D0, "D1": D1})

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

## 3. EXPERIMENT 3 - Real sample

### 3.1. bladder1

In [ ]:
ignore_censorship_bladder1 = bladder1
impute_censorship_bladder1 = complete_rd_csmap_dataset(bladder1)

#### 3.1.1. Work with censorship loglikelihood

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])
err_means_orig = err_corrs_orig = None
base_err_means_vs_emp = base_err_corrs_vs_emp = None
# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 0, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 100, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 500, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 1000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5000, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset
    
optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**4, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**5, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**6, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**7, work with censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="km",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])
err_means_orig = err_corrs_orig = None
base_err_means_vs_emp = base_err_corrs_vs_emp = None
# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

#### 3.1.2. Impute censorship

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10**7, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=3,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])
err_means_orig = err_corrs_orig = None
base_err_means_vs_emp = base_err_corrs_vs_emp = None
# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 0, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=0,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+0,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 100, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=100,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+100,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 500, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=500,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+500,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 1000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=1000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+1000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5000,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5000,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**4,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**4,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**5,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**5,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10**6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=5*10**6,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+5*10**6,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10**7, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_bladder1,
    order=4,
    formula="pepa",
    initialisations=10**4,
    distribution="gamma",
    parameter_A=1,
    parameter_B=0.5,
    cost_coefficient=10**7,
    method="naive",
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=5,
    seed=RANDOM_SEED+10**7,
    verbose=True,
    prescreen_mode="topk",
    top_k=20
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

optimizer.print_summary()

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])
err_means_orig = err_corrs_orig = None
base_err_means_vs_emp = base_err_corrs_vs_emp = None
# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

### 3.2. colorectal

In [ ]:
ignore_censorship_colorectal = colorectal
impute_censorship_colorectal = complete_rd_csmap_dataset(colorectal)

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10000, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 0, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 100, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 500, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 1000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=1000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10^4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10^4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10^5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10^5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10^6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 5*10^6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 3 ESTIMATION - cost_coefficient = 10^7, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=3,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**7,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 0, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 100, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10000, ignore censorship

optimizer = CSMAPOptimizer(
    dataset=ignore_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

In [ ]:
# For plotting, initialize storage
sweep = {
    "costs":   [],
    "scores":  [],
    "means":   [],
    "corrs":   [],
    "params":  [],
    "dataset": None
}

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 0, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=0,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 100, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 500, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=500,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 1000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=100,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5000, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5000,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10^4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10^4, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**4,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10^5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10^5, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**5,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10^6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 5*10^6, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=5*10**6,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# DIMENSION 4 ESTIMATION - cost_coefficient = 10^7, impute censorship

optimizer = CSMAPOptimizer(
    dataset=impute_censorship_colorectal,
    order=4,
    formula="pepa",
    initialisations=20,
    gamma_shape=2,
    gamma_scale=0.5,
    cost_coefficient=10**7,
    lower_bound_eps=1e-300,
    loglike_eps=1e-16,
    max_scipy_iter=20,
    seed=RANDOM_SEED,
    verbose=True
)

optimizer.print_summary()

# PLOTTING
# ---- Append to sweep store ----
a0h = optimizer.best_params["alpha0"]
D0h = optimizer.best_params["D0"]
D1h = optimizer.best_params["D1"]

est_means = [
    float(theoretical_times_moments(a0h, D0h, D1h, 1, 0, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 2, 1, 1)),
    float(theoretical_times_moments(a0h, D0h, D1h, 3, 2, 1)),
]
est_corrs = [
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 2, 1)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 1, 3, 2)),
    float(theoretical_times_correlations(a0h, D0h, D1h, 2, 3, 2)),
]

sweep["costs"].append(optimizer.cost_coefficient)
sweep["scores"].append(optimizer.best_score)
sweep["means"].append(est_means)
sweep["corrs"].append(est_corrs)
sweep["params"].append((a0h, D0h, D1h))
if sweep["dataset"] is None:
    sweep["dataset"] = optimizer.dataset

In [ ]:
# PLOTTING
# ----- gather arrays and sort by cost for prettier lines -----
costs = np.asarray(sweep["costs"], dtype=float)
idx   = np.argsort(costs)
costs = costs[idx]

est_means_grid = np.asarray(sweep["means"], dtype=float)[idx]  # (G,3)
est_corrs_grid = np.asarray(sweep["corrs"], dtype=float)[idx]  # (G,3)

# x positions equally spaced: 0,1,2,...,G-1
x = np.arange(costs.size)
# pretty tick labels
xtick_labels = [("0" if c == 0 else f"{int(c):,}") for c in costs]

dataset = sweep["dataset"]
if dataset is None:
    raise RuntimeError("sweep['dataset'] is empty — append at least one optimizer before plotting.")

# ----- empirical targets (computed once from the dataset) -----
emp_means = np.array([
    float(empirical_conditional_times_moments(dataset, n=1, l=0, k=1)),
    float(empirical_conditional_times_moments(dataset, n=2, l=1, k=1)),
    float(empirical_conditional_times_moments(dataset, n=3, l=2, k=1)),
], dtype=float)

emp_corrs = np.array([
    float(empirical_conditional_times_correlations(dataset, 1, 2, 1)),
    float(empirical_conditional_times_correlations(dataset, 1, 3, 2)),
    float(empirical_conditional_times_correlations(dataset, 2, 3, 2)),
], dtype=float)

# ----- original/theoretical targets (optional) -----
try:
    orig_means = np.array([
        float(theoretical_times_moments(alpha0, D0, D1, 1, 0, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 2, 1, 1)),
        float(theoretical_times_moments(alpha0, D0, D1, 3, 2, 1)),
    ], dtype=float)
    orig_corrs = np.array([
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)),
        float(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)),
        float(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)),
    ], dtype=float)
    have_original = True
except NameError:
    orig_means = orig_corrs = None
    have_original = False

# ----- errors -----
err_means_emp = np.abs(est_means_grid - emp_means[None, :])
err_corrs_emp = np.abs(est_corrs_grid - emp_corrs[None, :])

if have_original:
    err_means_orig = np.abs(est_means_grid - orig_means[None, :])
    err_corrs_orig = np.abs(est_corrs_grid - orig_corrs[None, :])
    base_err_means_vs_emp = np.abs(orig_means - emp_means)  # for dotted baselines
    base_err_corrs_vs_emp = np.abs(orig_corrs - emp_corrs)
else:
    err_means_orig = err_corrs_orig = None
    base_err_means_vs_emp = base_err_corrs_vs_emp = None

# =================== PLOTS ===================

# 1) Means: |Ê - Ē| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_means_emp[:,0], label="|Ê[T1]-Ē[T1]|", color="red")
plt.plot(x, err_means_emp[:,1], label="|Ê[T2]-Ē[T2]|", color="green")
plt.plot(x, err_means_emp[:,2], label="|Ê[T3]-Ē[T3]|", color="blue")
if base_err_means_vs_emp is not None:
    plt.axhline(base_err_means_vs_emp[0], ls=":", color="red",   label="|E₀[T1]-Ē[T1]| baseline")
    plt.axhline(base_err_means_vs_emp[1], ls=":", color="green", label="|E₀[T2]-Ē[T2]| baseline")
    plt.axhline(base_err_means_vs_emp[2], ls=":", color="blue",  label="|E₀[T3]-Ē[T3]| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Means: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 2) Means: |Ê - E₀| vs cost
if err_means_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_means_orig[:,0], label="|Ê[T1]-E₀[T1]|", color="red")
    plt.plot(x, err_means_orig[:,1], label="|Ê[T2]-E₀[T2]|", color="green")
    plt.plot(x, err_means_orig[:,2], label="|Ê[T3]-E₀[T3]|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Means: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping mean-error vs original plot.")

# 3) Correlations: |ρ̂ - ρ̄| vs cost
plt.figure(figsize=(8,5))
plt.plot(x, err_corrs_emp[:,0], label="|ρ̂ 12-ρ̄ 12|", color="red")
plt.plot(x, err_corrs_emp[:,1], label="|ρ̂ 13-ρ̄ 13|", color="green")
plt.plot(x, err_corrs_emp[:,2], label="|ρ̂ 23-ρ̄ 23|", color="blue")
if base_err_corrs_vs_emp is not None:
    plt.axhline(base_err_corrs_vs_emp[0], ls=":", color="red",   label="|ρ₀12-ρ̄ 12| baseline")
    plt.axhline(base_err_corrs_vs_emp[1], ls=":", color="green", label="|ρ₀13-ρ̄ 13| baseline")
    plt.axhline(base_err_corrs_vs_emp[2], ls=":", color="blue",  label="|ρ₀23-ρ̄ 23| baseline")
plt.axhline(0.0, color="k", lw=0.8, alpha=0.4)
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("absolute error vs empirical")
plt.title("Correlations: absolute error vs empirical")
plt.legend(ncol=2)
plt.tight_layout()

# 4) Correlations: |ρ̂ - ρ₀| vs cost
if err_corrs_orig is not None:
    plt.figure(figsize=(8,5))
    plt.plot(x, err_corrs_orig[:,0], label="|ρ̂ 12-ρ₀12|", color="red")
    plt.plot(x, err_corrs_orig[:,1], label="|ρ̂ 13-ρ₀13|", color="green")
    plt.plot(x, err_corrs_orig[:,2], label="|ρ̂ 23-ρ₀23|", color="blue")
    plt.axhline(0.0, color="k", lw=0.8, alpha=0.4, ls=":")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("absolute error vs original")
    plt.title("Correlations: absolute error vs original parameters")
    plt.legend()
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping correlation-error vs original plot.")
    
# =================== EXTRA PLOTS: actual values, not absolute errors ===================    
# A1) Means: Ê[Tn] vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
plt.axhline(emp_means[0], ls=":", color="red",   label="Ē[T1] baseline")
plt.axhline(emp_means[1], ls=":", color="green", label="Ē[T2] baseline")
plt.axhline(emp_means[2], ls=":", color="blue",  label="Ē[T3] baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")


plt.ylabel("mean inter-event time")
plt.title("Means: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A2) Means: Ê[Tn] vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_means_grid[:,0], label="Ê[T1]", color="red")
    plt.plot(x, est_means_grid[:,1], label="Ê[T2]", color="green")
    plt.plot(x, est_means_grid[:,2], label="Ê[T3]", color="blue")
    plt.axhline(orig_means[0], ls=":", color="red",   label="E₀[T1] baseline")
    plt.axhline(orig_means[1], ls=":", color="green", label="E₀[T2] baseline")
    plt.axhline(orig_means[2], ls=":", color="blue",  label="E₀[T3] baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("mean inter-event time")
    plt.title("Means: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Means: actual vs original' plot.")

# A3) Correlations: ρ̂ vs cost, with empirical baselines (dotted)
plt.figure(figsize=(8,5))
plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
plt.axhline(emp_corrs[0], ls=":", color="red",   label="ρ̄ 12 baseline")
plt.axhline(emp_corrs[1], ls=":", color="green", label="ρ̄ 13 baseline")
plt.axhline(emp_corrs[2], ls=":", color="blue",  label="ρ̄ 23 baseline")
ax = plt.gca()
apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
plt.ylabel("correlation")
plt.title("Correlations: estimated vs empirical baselines")
plt.legend(ncol=2)
plt.tight_layout()

# A4) Correlations: ρ̂ vs cost, with original-parameter baselines (dotted)
if have_original:
    plt.figure(figsize=(8,5))
    plt.plot(x, est_corrs_grid[:,0], label="ρ̂ 12", color="red")
    plt.plot(x, est_corrs_grid[:,1], label="ρ̂ 13", color="green")
    plt.plot(x, est_corrs_grid[:,2], label="ρ̂ 23", color="blue")
    plt.axhline(orig_corrs[0], ls=":", color="red",   label="ρ₀12 baseline")
    plt.axhline(orig_corrs[1], ls=":", color="green", label="ρ₀13 baseline")
    plt.axhline(orig_corrs[2], ls=":", color="blue",  label="ρ₀23 baseline")
    ax = plt.gca()
    apply_dense_cost_ticks(ax, x, costs, rotation=45, fontsize=9, bottom=0.22)
    plt.xlabel("cost_coefficient (log-ish grid, equally spaced)")
    plt.ylabel("correlation")
    plt.title("Correlations: estimated vs original-parameter baselines")
    plt.legend(ncol=2)
    plt.tight_layout()
else:
    print("⚠️  No original parameters found → skipping 'Correlations: actual vs original' plot.")